# Step 9 – Refined grade 2 re-stratification (Section 5.7, Table 16)

LODO proliferation score, the four grade 2 splits and the continuous proliferation score as a prognostic factor.

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

In [ ]:
import pickle, numpy as np, pandas as pd, warnings
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
%run pipeline.ipynb

warnings.filterwarnings("ignore")
CG = list(pd.read_csv("common_genes.csv").iloc[:, 0])
C4 = ["GSE7390", "UPP", "MAINZ", "UNT"]
END = {"GSE7390": "rfs", "UPP": "rfs", "MAINZ": "dmfs", "UNT": "rfs"}
S = pickle.load(open("summary.pkl", "rb"))
Z, frames = {}, []
for c in C4:
    X, y, P = load(c, genes=CG, zscore=True)
    Z[c] = (X, y)
    e = END[c]
    df = pd.DataFrame({"cohort": c, "grade": y + 1, "time": P[f"t.{e}"].values / 365.25, "event": P[f"e.{e}"].values,
                       "age": P["age"].values / 10, "size_gt2": (P["size"].values > 2).astype(int), "er": P["er"].values,
                       "node": P["node"].values}, index=X.index)
    pr = S["A"][c]["proba_oof"]
    df["call_3class"] = np.where(pr[:, 2] > pr[:, 0], 1, 0)
    # binary HWNB final model (trained on this cohort's G1+G3) applied to G2
    fin = pickle.load(open(f"folds_bin_{c}/final.pkl", "rb"))
    df["call_binHWNB"] = np.nan
    df.iloc[fin["i2"], df.columns.get_loc("call_binHWNB")] = (fin["pG3"] > 0.5).astype(int)
    # proliferation score, within cohort: module = genes with Spearman >= 0.5 to AURKA in G1+G3 tumours
    i13 = np.where(y != 1)[0]; i2 = np.where(y == 1)[0]
    Xv = X.values
    r = pd.DataFrame(Xv[i13]).rank().corrwith(pd.Series(Xv[i13, list(X.columns).index("AURKA")]).rank()).values
    mod = np.where(r >= 0.5)[0]
    score = Xv[:, mod].mean(1)
    lr = LogisticRegression(class_weight="balanced").fit(score[i13, None], (y[i13] == 2).astype(int))
    df["call_prolif"] = np.nan
    df.iloc[i2, df.columns.get_loc("call_prolif")] = (lr.predict_proba(score[i2, None])[:, 1] > 0.5).astype(int)
    # label-free score on all tumours for prognosis (module from all tumours, no grade used)
    r_all = pd.DataFrame(Xv).rank().corrwith(pd.Series(Xv[:, list(X.columns).index("AURKA")]).rank()).values
    df["prolif_all"] = Xv[:, np.where(r_all >= 0.5)[0]].mean(1)
    df["prolif_all"] = (df["prolif_all"] - df["prolif_all"].mean()) / df["prolif_all"].std()
    frames.append(df)
D = pd.concat(frames)

### LODO for the proliferation score: module and threshold learned on three cohorts

In [ ]:
# ---- LODO for the proliferation score: module and threshold learned on three cohorts
D["call_prolif_lodo"] = np.nan
lodo = {}
for held in C4:
    tr = [c for c in C4 if c != held]
    Xtr = pd.concat([Z[c][0] for c in tr]); ytr = np.concatenate([Z[c][1] for c in tr])
    r = Xtr.rank().corrwith(Xtr["AURKA"].rank())
    mod = list(r[r >= 0.5].index)
    s_tr = Xtr[mod].mean(1).values
    m13 = ytr != 1
    lr = LogisticRegression(class_weight="balanced").fit(s_tr[m13, None], (ytr[m13] == 2).astype(int))
    Xte, yte = Z[held]
    s_te = Xte[mod].mean(1).values
    t13 = yte != 1
    p = lr.predict_proba(s_te[:, None])[:, 1]
    yb = (yte[t13] == 2).astype(int)
    lodo[held] = {"n13": int(t13.sum()), "module": len(mod), "auc": roc_auc_score(yb, p[t13]),
                  "f1": f1_score(yb, (p[t13] > 0.5).astype(int), average="macro"), "mcc": matthews_corrcoef(yb, (p[t13] > 0.5).astype(int))}
    idx = D.index[(D.cohort == held)]
    vals = np.where(yte == 1, (p > 0.5).astype(float), np.nan)
    D.loc[idx, "call_prolif_lodo"] = vals
    print("LODO prolif", held, {k: round(v, 3) if isinstance(v, float) else v for k, v in lodo[held].items()})

D = D.dropna(subset=["time", "event"]); D["event"] = D["event"].astype(int)
G2 = D[D.grade == 2]
res = {}
for call in ["call_3class", "call_binHWNB", "call_prolif", "call_prolif_lodo"]:
    d = G2.dropna(subset=[call]).copy(); d["g3like"] = d[call].astype(int)
    lr_ = logrank_test(d.time[d.g3like == 1], d.time[d.g3like == 0], d.event[d.g3like == 1], d.event[d.g3like == 0])
    km = {}
    for g in [0, 1]:
        k = KaplanMeierFitter().fit(d.time[d.g3like == g], d.event[d.g3like == g])
        km[g] = (int((d.g3like == g).sum()), round(float(k.survival_function_at_times(5).iloc[0]) * 100, 1), round(float(k.survival_function_at_times(10).iloc[0]) * 100, 1))
    dd = d[["time", "event", "g3like", "er", "size_gt2", "age", "node", "cohort"]].dropna()
    m = CoxPHFitter().fit(dd, "time", "event", strata=["cohort"])
    s = m.summary.loc["g3like"]
    ci = concordance_index(d.time, -d.g3like, d.event)
    res[call] = {"logrank": lr_.p_value, "km": km, "hr": (s["exp(coef)"], s["exp(coef) lower 95%"], s["exp(coef) upper 95%"], s["p"]), "n": len(dd), "ev": int(dd.event.sum()), "c": ci}
    print(call, "logrank p=%.3g" % lr_.p_value, km, "adjHR %.2f (%.2f-%.2f) p=%.3g" % res[call]["hr"], "n", len(dd), "ev", int(dd.event.sum()))
# agreement between calls
for a, b in [("call_3class", "call_prolif"), ("call_binHWNB", "call_prolif"), ("call_prolif", "call_prolif_lodo")]:
    d = G2.dropna(subset=[a, b]); print("agreement", a, b, round((d[a] == d[b]).mean(), 3))
# continuous proliferation score as prognostic factor in all tumours
cidx = {c: concordance_index(D[D.cohort == c].time, -D[D.cohort == c].prolif_all, D[D.cohort == c].event) for c in C4}
dd = D[["time", "event", "prolif_all", "grade", "er", "size_gt2", "age", "node", "cohort"]].dropna().copy()
dd["g3"] = (dd.grade == 3).astype(int); dd["g2"] = (dd.grade == 2).astype(int)
m = CoxPHFitter().fit(dd.drop(columns=["grade"]), "time", "event", strata=["cohort"])
s = m.summary
print("C-index prolif_all", {c: round(v, 3) for c, v in cidx.items()})
for v in ["prolif_all", "g2", "g3", "er", "size_gt2", "age", "node"]:
    print("  multi", v, "%.2f (%.2f-%.2f) p=%.3g" % (s.loc[v, "exp(coef)"], s.loc[v, "exp(coef) lower 95%"], s.loc[v, "exp(coef) upper 95%"], s.loc[v, "p"]))
res["prolif_cidx"] = cidx; res["prolif_multi"] = {v: tuple(s.loc[v, ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]) for v in s.index}
res["lodo"] = lodo; res["n_multi"] = (len(dd), int(dd.event.sum()))
# KM curves for the LODO proliferation split
curves = {}
d = G2.dropna(subset=["call_prolif_lodo"])
for g in [0, 1]:
    k = KaplanMeierFitter().fit(d.time[d.call_prolif_lodo == g], d.event[d.call_prolif_lodo == g])
    sf = k.survival_function_; last = None; pts = []
    for t, v in zip(sf.index, sf.iloc[:, 0]):
        if t > 10: break
        v = round(float(v), 4)
        if v != last: pts.append((round(float(t), 3), v)); last = v
    curves[g] = pts
res["curves_lodo"] = curves
pickle.dump(res, open("survival_refined.pkl", "wb"))